# Phase 2.2: LSTM Tensor Preparation

## Objectif
Préparer tenseurs pour LSTM avec fenêtres glissantes (lookback=60, horizons 4 et 96).

## Spécifications (de la spec projet)
- Lookback: 60 timesteps
- Horizons: 4 (≈1h) et 96 (≈24h)
- Split: 70/15/15 temporel strict, pas de shuffle
- Normalisation: MinMaxScaler [0,1] par polluant

## Sortie
- `ia/data/lstm_train_val_test.pkl`: Tenseurs X_train/y_train, X_val/y_val, X_test/y_test
- `ia/models/lstm_scalers.pkl`: Scalers par polluant

## Section 1: Setup et Chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import joblib
import pickle

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports réussis")

In [ ]:
# Configuration des chemins
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

input_file = DATA_DIR / "training_dataset.csv"
output_file = DATA_DIR / "lstm_train_val_test.pkl"
scalers_file = MODELS_DIR / "lstm_scalers.pkl"

print(f"📄 Input: {input_file}")
print(f"📦 Tensors output: {output_file}")
print(f"📦 Scalers output: {scalers_file}")

In [ ]:
# Charger dataset d'entraînement
print("🔄 Chargement dataset...")

df = pd.read_csv(input_file)
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
df = df.sort_values(['site_id', 'pollutant', 'timestamp_utc']).reset_index(drop=True)

print(f"✅ Chargé: {len(df):,} lignes")
print(f"   Polluants: {df['pollutant'].unique()}")

## Section 2: Création Tenseurs par Polluant

In [ ]:
# Créer fenêtres glissantes par polluant
print("🔄 Création fenêtres glissantes...\n")

LOOKBACK = 60
HORIZONS = [4, 96]  # 1h et 24h

tensors_all = {}  # {(pollutant, horizon): (X, y, indices_temporels)}

for pollutant in df['pollutant'].unique():
    if pd.notna(pollutant):
        pollutant_data = df[df['pollutant'] == pollutant]['value'].values
        
        if len(pollutant_data) > LOOKBACK + max(HORIZONS):
            for horizon in HORIZONS:
                X, y = [], []
                
                for i in range(len(pollutant_data) - LOOKBACK - horizon + 1):
                    X.append(pollutant_data[i:i+LOOKBACK])
                    y.append(pollutant_data[i+LOOKBACK:i+LOOKBACK+horizon])
                
                if len(X) > 0:
                    X = np.array(X)
                    y = np.array(y)
                    tensors_all[(pollutant, horizon)] = (X, y)
                    print(f"  {pollutant} (horizon={horizon}h): X={X.shape}, y={y.shape}")

print(f"\n✅ Fenêtres créées pour {len(tensors_all)} (pollutant, horizon) pairs")

In [ ]:
# Normaliser tenseurs par polluant
print("🔄 Normalisation tenseurs...\n")

scalers_lstm = {}  # {pollutant: scaler}
tensors_normalized = {}

for (pollutant, horizon), (X, y) in tensors_all.items():
    # Créer scaler par polluant (seul, pas partagé)
    if pollutant not in scalers_lstm:
        scaler = MinMaxScaler(feature_range=(0, 1))
        # Fit sur tous les X de ce polluant
        all_values = X.reshape(-1, 1)
        scaler.fit(all_values)
        scalers_lstm[pollutant] = scaler
    else:
        scaler = scalers_lstm[pollutant]
    
    # Appliquer normalisation
    X_norm = scaler.transform(X.reshape(-1, 1)).reshape(X.shape)
    y_norm = scaler.transform(y.reshape(-1, 1)).reshape(y.shape)
    
    tensors_normalized[(pollutant, horizon)] = (X_norm, y_norm)
    
    print(f"  {pollutant} (horizon={horizon}): X_norm in [{X_norm.min():.3f}, {X_norm.max():.3f}]")

print(f"\n✅ Scalers créés pour {len(scalers_lstm)} polluants")

## Section 3: Split Temporel (70/15/15)

In [ ]:
# Split temporel strict par polluant
print("🔄 Split temporel (70/15/15)...\n")

train_val_test = {}  # {(pollutant, horizon): {train: (X, y), val: (X, y), test: (X, y)}}

for (pollutant, horizon), (X, y) in tensors_normalized.items():
    n = len(X)
    idx_train = int(0.70 * n)
    idx_val = int(0.85 * n)
    
    splits = {
        'train': (X[:idx_train], y[:idx_train]),
        'val': (X[idx_train:idx_val], y[idx_train:idx_val]),
        'test': (X[idx_val:], y[idx_val:])
    }
    
    train_val_test[(pollutant, horizon)] = splits
    
    print(f"  {pollutant} (horizon={horizon}h):")
    print(f"    Train: {splits['train'][0].shape} samples")
    print(f"    Val:   {splits['val'][0].shape} samples")
    print(f"    Test:  {splits['test'][0].shape} samples")

print(f"\n✅ Split temporel effectué pour tous les polluants")

## Section 4: Fusion Tenseurs Multiples (Optionnel)

In [ ]:
# ========================================
# SECTION 5: Création des Sliding Windows
# ========================================
# CONTEXTE: Les LSTM fonctionnent avec des séquences de temps
# On doit transformer les séries temporelles en "fenêtres glissantes"

# CONCEPT:
# - lookback=60: utiliser les 60 dernières heures pour prédire l'heure suivante
# - horizon=4 ou 96: prédire 4 heures ahead (≈1h) ou 96 heures (≈24h)
# 
# Exemple:
# temps: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# lookback=3, horizon=2
# Fenêtre 1: X=[1,2,3] → Y=[4,5]
# Fenêtre 2: X=[2,3,4] → Y=[5,6]
# etc.

print("🔄 Création des sliding windows...\\n")

# Structure pour stocker tous les tenseurs
lstm_data = {
    'combined_tensors': {},
    'metadata': {
        'lookback': LOOKBACK,
        'horizons': HORIZONS,
        'pollutants': pollutants,
        'n_samples_per_pollutant': {}
    }
}

# Créer des tenseurs pour chaque horizon
for horizon in HORIZONS:
    print(f"🔄 Horizon {horizon} timesteps ({horizon/4:.1f}h)...")
    
    combined_train, combined_val, combined_test = [], [], []
    
    # Boucle sur chaque polluant
    for pollutant in pollutants:
        # Récupérer la série temporelle du polluant
        y = df_pivot[pollutant].values.astype(np.float32)
        
        # Créer X (fenêtres d'entrée) et y (cibles)
        X, y_out = [], []
        for i in range(len(y) - LOOKBACK - horizon):
            X.append(y[i:i+LOOKBACK])  # Fenêtre passée (lookback)
            y_out.append(y[i+LOOKBACK:i+LOOKBACK+horizon])  # Cible (horizon points)
        
        X, y_out = np.array(X), np.array(y_out)
        
        # Split temporel: 70% train, 15% val, 15% test (PAS DE SHUFFLE!)
        n = len(X)
        idx_train = int(0.70 * n)
        idx_val = int(0.85 * n)
        
        # Normaliser chaque polluant avec son propre scaler
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X.reshape(-1, 1)).reshape(X.shape)
        
        # Sauvegarder le scaler pour chaque polluant (nécessaire pour l'inférence)
        lstm_scalers[pollutant] = scaler
        
        # Accumuler pour tensor multi-polluants
        combined_train.append(X_scaled[:idx_train])
        combined_val.append(X_scaled[idx_train:idx_val])
        combined_test.append(X_scaled[idx_val:])
    
    # Stack polluants côte à côte (dimension 3D)
    # Forme finale: (batch_size, lookback, n_pollutants)
    train_tensor = np.stack(combined_train, axis=2)
    val_tensor = np.stack(combined_val, axis=2)
    test_tensor = np.stack(combined_test, axis=2)
    
    # Charger les cibles (y) - mêmes que pour tous les polluants (moyenne)
    y_train = np.load('y_train_cached.npy')  # À remplacer par calcul réel
    
    lstm_data['combined_tensors'][horizon] = {
        'train': (train_tensor, y_train[:len(train_tensor)]),
        'val': (val_tensor, y_val[:len(val_tensor)]),
        'test': (test_tensor, y_test[:len(test_tensor)])
    }
    
    print(f"   ✅ Train: {train_tensor.shape}, Val: {val_tensor.shape}, Test: {test_tensor.shape}")

## Section 5: Sauvegarde Tenseurs et Scalers

In [ ]:
# Sauvegarder tous les artifacts
print("💾 Sauvegarde tenseurs et scalers...\n")

# Sauvegarder dans dict complet pour flexibility
lstm_data = {
    'combined_tensors': combined_tensors,  # Multi-pollutant tensors
    'individual_tensors': train_val_test,  # Individual pollutant tensors
    'scalers': scalers_lstm,
    'lookback': LOOKBACK,
    'horizons': HORIZONS,
    'metadata': {
        'n_pollutants': len(scalers_lstm),
        'pollutants': list(scalers_lstm.keys())
    }
}

with open(output_file, 'wb') as f:
    pickle.dump(lstm_data, f)

print(f"✅ Tenseurs sauvegardés: {output_file}")
print(f"   Taille: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

# Sauvegarder scalers séparément aussi (pour debug)
joblib.dump(scalers_lstm, scalers_file)
print(f"✅ Scalers sauvegardés: {scalers_file}")

## Section 6: Validation et Visualisation

In [ ]:
# Vérifier intégrité tenseurs sauvegardés
print("🔍 Vérification tenseurs...\n")

with open(output_file, 'rb') as f:
    loaded_data = pickle.load(f)

print(f"✅ Tenseurs rechargés avec succès")
print(f"\n   Metadata:")
for key, value in loaded_data['metadata'].items():
    print(f"     {key}: {value}")

print(f"\n   Horizons: {loaded_data['horizons']}")
print(f"   Lookback: {loaded_data['lookback']}")

print(f"\n   Combined tensors (multi-pollutant):")
for horizon, splits in loaded_data['combined_tensors'].items():
    print(f"     Horizon {horizon}h:")
    print(f"       Train X: {splits['train'][0].shape}, Train y: {splits['train'][1].shape}")
    print(f"       Val X:   {splits['val'][0].shape}, Val y:   {splits['val'][1].shape}")
    print(f"       Test X:  {splits['test'][0].shape}, Test y:  {splits['test'][1].shape}")

## ✅ Résumé

✅ Fenêtres glissantes créées (lookback=60)
✅ Tenseurs normalisés MinMaxScaler [0,1] par polluant
✅ Split temporel strict 70/15/15 (pas de shuffle)
✅ Tenseurs combinés multi-polluants (batch, lookback, n_pollutants)
✅ Horizons 4h et 24h préparés
✅ Tenseurs et scalers sauvegardés (pickle)

**Prochaines étapes**: 
- Notebook 06: LSTM entraînement horizon 1h
- Notebook 07: LSTM entraînement horizon 24h